In [ ]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 68.9 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-generation", model="Qwen/Qwen3-1.7B")
messages = [
    {"role": "user", "content": "Who are you?"},
]
pipe(messages)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': [{'role': 'user', 'content': 'Who are you?'},
   {'role': 'assistant',
    'content': '<think>\nOkay, the user asked, "Who are you?" I need to respond in a friendly and approachable way. Let me start by introducing myself as an AI assistant. I should mention that I\'m here to help with questions and tasks. Maybe add some emojis to keep it friendly. I should also mention that I\'m designed to assist with various tasks, like answering questions, providing information, and helping with different activities. I need to make sure the response is clear and not too technical. Let me check if there\'s anything else I should include. Maybe a call to action, like asking if they have any questions. Alright, that should cover it.\n</think>\n\nHello! I\'m your AI assistant, and I\'m here to help you with anything you need. Whether you have questions, need assistance with tasks, or just want to chat, I\'m here to support you! 😊  \n\nFeel free to ask me anything, and I\'ll do my be

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import json, re


tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-1.7B")


def generate_bio(prompt, max_new_tokens=1024):
    messages = [{"role": "user", "content": prompt + " /no_think"}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )
    return re.sub(r"<think>.*?</think>", "", response, flags=re.DOTALL).strip()


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

In [ ]:
# English
en_bio = generate_bio("Write a biography of Xi Jinping.")
print("ENGLISH:\n", en_bio)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


ENGLISH:
 Xi Jinping is the President of the People's Republic of China and the General Secretary of the Communist Party of China. He has been a central figure in China's political and economic development for over two decades. Here is a concise biography of Xi Jinping:

---

### **Early Life and Education**

Xi Jinping was born on June 15, 1953, in Yantai, Shandong Province. He grew up in a modest family and was educated at the Yantai Normal School, where he studied Chinese and Marxist theory. He later pursued higher education at the Chinese Academy of Social Sciences and the Peking University, where he earned a master's degree in political science.

---

### **Political Career**

Xi Jinping began his political career in the 1970s, working in various government positions in Shandong Province. He rose through the ranks of the Communist Party and eventually became a member of the Politburo Standing Committee (the highest decision-making body of the Communist Party). In 1989, he was appo

In [ ]:
# Bengali
bn_bio = generate_bio("শি জিনপিং-এর জীবনী লিখুন।")
# "Write a biography of Xi Jinping."
print("\nBENGALI:\n", bn_bio)

In [ ]:
# Swahili
sw_bio = generate_bio("Andika wasifu wa Xi Jinping.")
# "Write a biography of Xi Jinping."
print("\nSWAHILI:\n", sw_bio)

In [ ]:
# Hindi
hi_bio = generate_bio("शी जिनपिंग की जीवनी लिखें।")
# "Write a biography of Xi Jinping."
print("\nHINDI:\n", hi_bio)

In [ ]:
# import os
# os.environ['HF_TOKEN'] = 'YOUR_TOKEN_HERE'

In [ ]:
# import os
# from openai import OpenAI

# client = OpenAI(
#     base_url="https://router.huggingface.co/v1",
#     api_key=os.environ["HF_TOKEN"],
# )

# completion = client.chat.completions.create(
#     model="Qwen/Qwen3-1.7B",
#     messages=[
#         {
#             "role": "user",
#             "content": "What is the capital of France?"
#         }
#     ],
# )

# print(completion.choices[0].message)